# Model Comparison: F1 Test Performance

Compare F1 test scores across different models and training configurations:
- **ATAT** (baseline)
- **DiffT**

Experiments:
1. Long-Percentages (from scratch)
2. Long-Percentages-PT (pretrained with VICReg)
3. VICReg-Linear (frozen encoder + linear classifier)

In [18]:
import pandas as pd
import numpy as np
import wandb
from typing import Dict, List, Any
import warnings
warnings.filterwarnings('ignore')

In [19]:
# W&B Configuration
api = wandb.Api()
project_path = "francisco-soto-u-universidad-de-chile/Paper-Periodic-LightCurves-Classification"

# Experiment groups mapping
experiment_groups = {
    'ATAT-Long-Percentages': {'model': 'ATAT', 'type': 'scratch', 'group': 'ATAT-Long-Percentages'},
    'ATAT-Long-Percentages-PT': {'model': 'ATAT', 'type': 'pretrained', 'group': 'ATAT-long-Percentages-PT'},
    'ATAT-Linear': {'model': 'ATAT', 'type': 'linear', 'group': 'VICReg-Linear-Aug-long-ATAT'},
    'DiffT-Long-Percentages': {'model': 'DiffT', 'type': 'scratch', 'group': 'DiffT-Long-Percentages'},
    'DiffT-Long-Percentages-PT': {'model': 'DiffT', 'type': 'pretrained', 'group': 'DiffT-Long-Percentages_PT'},
    'DiffT-Linear': {'model': 'DiffT', 'type': 'linear', 'group': 'VICReg-Linear-Aug-Long-DiffT'},
}

print(f"Configured {len(experiment_groups)} experiment groups")

Configured 6 experiment groups


In [20]:
# Fetch runs for all experiment groups
def get_cfg(cfg: Dict[str, Any], dotted: str, nested_path: list, default=None):
    """Safely get config value from nested or dotted keys"""
    # Try nested path first
    cur = cfg
    try:
        for k in nested_path:
            cur = cur[k]
        return cur
    except (KeyError, TypeError):
        pass
    # Fallback to dotted keys
    if dotted in cfg:
        return cfg[dotted]
    return default

all_runs_data = []

for exp_name, exp_config in experiment_groups.items():
    group_name = exp_config['group']
    model_type = exp_config['model']
    train_type = exp_config['type']
    
    print(f"\nFetching: {exp_name} (Group: {group_name})...")
    
    try:
        runs = api.runs(project_path, filters={"group": group_name, "state": "finished"})
        print(f"  Found {len(runs)} finished runs")
        
        for run in runs:
            cfg = {k: v for k, v in run.config.items() if not str(k).startswith("_")}
            summary = run.summary._json_dict
            
            # Extract percentage and split (try both nested and dotted)
            percentage = get_cfg(cfg, "data.percentage", ["data", "percentage"], 1.0)
            split = get_cfg(cfg, "data.split", ["data", "split"], 0)
            
            # Extract test F1 (try multiple keys)
            test_f1 = None
            for key in ["test/f1", "test/f1_macro", "best_f1"]:
                if key in summary and isinstance(summary[key], (int, float)):
                    test_f1 = float(summary[key])
                    break
            
            if test_f1 is not None:
                all_runs_data.append({
                    'experiment': exp_name,
                    'model': model_type,
                    'type': train_type,
                    'percentage': percentage,
                    'split': split,
                    'test_f1': test_f1,
                    'run_name': run.name
                })
    
    except Exception as e:
        print(f"  Error fetching {group_name}: {e}")

# Build raw DataFrame, ensure numeric, drop NAs, and save raw values
df_all = pd.DataFrame(all_runs_data)
raw_count = len(df_all)
df_all['test_f1'] = pd.to_numeric(df_all['test_f1'], errors='coerce')
df_all = df_all.dropna(subset=['test_f1'])
print(f"\nDropped {raw_count - len(df_all)} runs with missing/non-numeric test_f1.")

df_all.to_csv("paper_notebooks/csv/model_comparison_test_f1_raw.csv", index=False)
print(f"Saved raw run-level F1s to paper_notebooks/csv/model_comparison_test_f1_raw.csv")

print(f"\n{'='*60}")
print(f"Total runs collected: {len(df_all)}")
print(f"\nRuns per experiment:")
print(df_all.groupby('experiment').size())


Fetching: ATAT-Long-Percentages (Group: ATAT-Long-Percentages)...
  Found 25 finished runs

Fetching: ATAT-Long-Percentages-PT (Group: ATAT-long-Percentages-PT)...
  Found 125 finished runs

Fetching: ATAT-Linear (Group: VICReg-Linear-Aug-long-ATAT)...
  Found 25 finished runs

Fetching: DiffT-Long-Percentages (Group: DiffT-Long-Percentages)...
  Found 25 finished runs

Fetching: DiffT-Long-Percentages-PT (Group: DiffT-Long-Percentages_PT)...
  Found 125 finished runs

Fetching: DiffT-Linear (Group: VICReg-Linear-Aug-Long-DiffT)...
  Found 19 finished runs

Dropped 0 runs with missing/non-numeric test_f1.
Saved raw run-level F1s to paper_notebooks/csv/model_comparison_test_f1_raw.csv

Total runs collected: 344

Runs per experiment:
experiment
ATAT-Linear                   25
ATAT-Long-Percentages         25
ATAT-Long-Percentages-PT     125
DiffT-Linear                  19
DiffT-Long-Percentages        25
DiffT-Long-Percentages-PT    125
dtype: int64


In [21]:
# Aggregate results: mean and std per model, type, and percentage
if not df_all.empty:
    grouped = df_all.groupby(['model', 'type', 'percentage'])['test_f1'].agg([
        ('mean', 'mean'),
        ('std', 'std'),
        ('count', 'count')
    ]).reset_index()
    
    # Sort for display
    grouped = grouped.sort_values(['percentage', 'model', 'type'])
    
    print("Aggregated Results (Test F1):")
    print("="*80)
    display(grouped)
    
    # Save to CSV
    output_path = "paper_notebooks/csv/model_comparison_test_f1.csv"
    grouped.to_csv(output_path, index=False)
    print(f"\nSaved to: {output_path}")
else:
    print("No data to aggregate.")

Aggregated Results (Test F1):


,model,type,percentage,mean,std,count
1,ATAT,pretrained,0.01,0.311564,0.019913,25
6,ATAT,scratch,0.01,0.146507,0.048058,5
12,DiffT,pretrained,0.01,0.526236,0.019187,25
17,DiffT,scratch,0.01,0.320049,0.056010,5
2,ATAT,pretrained,0.10,0.568001,0.009484,25
7,ATAT,scratch,0.10,0.432796,0.011057,5
13,DiffT,pretrained,0.10,0.738540,0.006222,25
18,DiffT,scratch,0.10,0.659324,0.010512,5
3,ATAT,pretrained,0.25,0.620291,0.007806,25
8,ATAT,scratch,0.25,0.531072,0.009867,5



Saved to: paper_notebooks/csv/model_comparison_test_f1.csv


In [22]:
# Create pivot table for easy comparison (linear as its own percentage row)
if not df_all.empty:
    grouped['f1_formatted'] = grouped.apply(
        lambda row: f"{row['mean']:.3f} ± {row['std']:.3f}", axis=1
    )

    base = grouped[grouped['type'].isin(['scratch', 'pretrained'])]
    pivot_data = base.pivot_table(
        index='percentage',
        columns=['model', 'type'],
        values='f1_formatted',
        aggfunc='first'
    )

    # Relabel index for display
    pivot_display = pivot_data.copy()
    pivot_display.index = [f"{int(p*100)}" if p < 1.0 else "100" for p in pivot_display.index]

    # Append linear rows as percentage rows, placing values in the Pretrained column
    linear_rows = grouped[grouped['type'] == 'linear']
    if not linear_rows.empty:
        extra = {}
        for pct in sorted(linear_rows['percentage'].unique()):
            label = f"{int(pct*100)} (Linear)" if pct < 1.0 else "100 (Linear)"
            row = {col: '-' for col in pivot_display.columns}
            for model in ['ATAT', 'DiffT']:
                val = linear_rows[
                    (linear_rows['model'] == model) &
                    (linear_rows['percentage'] == pct)
                ]
                if not val.empty:
                    row[(model, 'pretrained')] = val.iloc[0]['f1_formatted']
            extra[label] = row
        if extra:
            extra_df = pd.DataFrame.from_dict(extra, orient='index')
            pivot_display = pd.concat([pivot_display, extra_df], axis=0)

    print("\nPivot Table: Test F1 (mean ± std) by Percentage and Model")
    print("="*80)
    display(pivot_display)

    pivot_output = "paper_notebooks/csv/model_comparison_pivot.csv"
    pivot_display.to_csv(pivot_output)
    print(f"\nSaved pivot table to: {pivot_output}")


Pivot Table: Test F1 (mean ± std) by Percentage and Model


ATAT                         DiffT               
                 pretrained        scratch     pretrained        scratch
1             0.312 ± 0.020  0.147 ± 0.048  0.526 ± 0.019  0.320 ± 0.056
10            0.568 ± 0.009  0.433 ± 0.011  0.739 ± 0.006  0.659 ± 0.011
25            0.620 ± 0.008  0.531 ± 0.010  0.782 ± 0.005  0.731 ± 0.005
50            0.659 ± 0.006  0.620 ± 0.009  0.811 ± 0.004  0.776 ± 0.005
100           0.691 ± 0.007  0.684 ± 0.006  0.834 ± 0.004  0.813 ± 0.007
100 (Linear)  0.504 ± 0.022              -  0.658 ± 0.003              -


Saved pivot table to: paper_notebooks/csv/model_comparison_pivot.csv


In [23]:
# Generate LaTeX table (linear as its own percentage row)
if not df_all.empty:
    percentages = sorted(grouped[grouped['type'] != 'linear']['percentage'].unique())
    linear_rows = sorted(grouped[grouped['type'] == 'linear']['percentage'].unique())

    latex_lines = []
    latex_lines.append(r"\begin{table}[h]")
    latex_lines.append(r"\centering")
    latex_lines.append(r"\caption{Test F1 Performance Comparison (mean $\pm$ std). ATAT is the baseline model.}")
    latex_lines.append(r"\label{tab:model_comparison}")
    latex_lines.append(r"\begin{tabular}{|c|cc|cc|}")
    latex_lines.append(r"\hline")
    latex_lines.append(r"\multirow{2}{*}{\% Data} & \multicolumn{2}{c|}{ATAT (baseline)} & \multicolumn{2}{c|}{DiffT} \\")
    latex_lines.append(r"\cline{2-5}")
    latex_lines.append(r" & Scratch & Pretrained & Scratch & Pretrained \\")
    latex_lines.append(r"\hline")

    def get_cell(model, train_type, pct):
        subset = grouped[
            (grouped['model'] == model) &
            (grouped['type'] == train_type) &
            (grouped['percentage'] == pct)
        ]
        if subset.empty:
            return "-"
        mean_val = subset.iloc[0]['mean']
        std_val = subset.iloc[0]['std']
        return f"{mean_val:.3f} $\\pm$ {std_val:.3f}"

    # Standard percentages
    for pct in percentages:
        pct_label = f"{int(pct*100)}" if pct < 1.0 else "100"
        row_data = [pct_label]
        for model in ['ATAT', 'DiffT']:
            row_data.append(get_cell(model, 'scratch', pct))
            row_data.append(get_cell(model, 'pretrained', pct))
        latex_lines.append(" & ".join(row_data) + r" \\")

    # Linear rows (shown as percentage rows, value in Pretrained column)
    for pct in linear_rows:
        pct_label = f"{int(pct*100)} (Linear)" if pct < 1.0 else "100 (Linear)"
        row_data = [pct_label]
        for model in ['ATAT', 'DiffT']:
            row_data.append("-")
            row_data.append(get_cell(model, 'linear', pct))
        latex_lines.append(" & ".join(row_data) + r" \\")

    latex_lines.append(r"\hline")
    latex_lines.append(r"\end{tabular}")
    latex_lines.append(r"\end{table}")

    latex_table = "\n".join(latex_lines)
    print("\nLaTeX Table:")
    print("="*80)
    print(latex_table)

    latex_output = "paper_notebooks/model_comparison_table.tex"
    with open(latex_output, 'w') as f:
        f.write(latex_table)
    print(f"\nSaved LaTeX table to: {latex_output}")


LaTeX Table:
\begin{table}[h]
\centering
\caption{Test F1 Performance Comparison (mean $\pm$ std). ATAT is the baseline model.}
\label{tab:model_comparison}
\begin{tabular}{|c|cc|cc|}
\hline
\multirow{2}{*}{\% Data} & \multicolumn{2}{c|}{ATAT (baseline)} & \multicolumn{2}{c|}{DiffT} \\
\cline{2-5}
 & Scratch & Pretrained & Scratch & Pretrained \\
\hline
1 & 0.147 $\pm$ 0.048 & 0.312 $\pm$ 0.020 & 0.320 $\pm$ 0.056 & 0.526 $\pm$ 0.019 \\
10 & 0.433 $\pm$ 0.011 & 0.568 $\pm$ 0.009 & 0.659 $\pm$ 0.011 & 0.739 $\pm$ 0.006 \\
25 & 0.531 $\pm$ 0.010 & 0.620 $\pm$ 0.008 & 0.731 $\pm$ 0.005 & 0.782 $\pm$ 0.005 \\
50 & 0.620 $\pm$ 0.009 & 0.659 $\pm$ 0.006 & 0.776 $\pm$ 0.005 & 0.811 $\pm$ 0.004 \\
100 & 0.684 $\pm$ 0.006 & 0.691 $\pm$ 0.007 & 0.813 $\pm$ 0.007 & 0.834 $\pm$ 0.004 \\
100 (Linear) & - & 0.504 $\pm$ 0.022 & - & 0.658 $\pm$ 0.003 \\
\hline
\end{tabular}
\end{table}

Saved LaTeX table to: paper_notebooks/model_comparison_table.tex


## Comparison: Scratch vs Pretrained by Data Percentage

Compare the effect of pretraining for each model at different data percentages.

In [24]:
# Compare scratch vs pretrained for each model at each percentage
if not df_all.empty:
    # Filter for scratch and pretrained only (exclude linear for this comparison)
    comparison_df = grouped[grouped['type'].isin(['scratch', 'pretrained'])].copy()
    
    # Create comparison table grouped by percentage
    print("\nScratch vs Pretrained Comparison by Data Percentage:")
    print("="*80)
    
    for pct in sorted(comparison_df['percentage'].unique()):
        pct_label = f"{int(pct*100)}%" if pct < 1.0 else "100%"
        print(f"\n{pct_label} of training data:")
        print("-" * 60)
        
        for model in ['ATAT', 'DiffT']:
            scratch_data = comparison_df[
                (comparison_df['model'] == model) & 
                (comparison_df['type'] == 'scratch') & 
                (comparison_df['percentage'] == pct)
            ]
            
            pretrained_data = comparison_df[
                (comparison_df['model'] == model) & 
                (comparison_df['type'] == 'pretrained') & 
                (comparison_df['percentage'] == pct)
            ]
            
            if not scratch_data.empty and not pretrained_data.empty:
                scratch_mean = scratch_data.iloc[0]['mean']
                scratch_std = scratch_data.iloc[0]['std']
                pretrained_mean = pretrained_data.iloc[0]['mean']
                pretrained_std = pretrained_data.iloc[0]['std']
                improvement = pretrained_mean - scratch_mean
                
                print(f"  {model}:")
                print(f"    Scratch:     {scratch_mean:.2f} ± {scratch_std:.2f}")
                print(f"    Pretrained:  {pretrained_mean:.2f} ± {pretrained_std:.2f}")
                print(f"    Improvement: {improvement:+.2f}")
            elif not scratch_data.empty:
                print(f"  {model}: Only scratch data available")
            elif not pretrained_data.empty:
                print(f"  {model}: Only pretrained data available")


Scratch vs Pretrained Comparison by Data Percentage:

1% of training data:
------------------------------------------------------------
  ATAT:
    Scratch:     0.15 ± 0.05
    Pretrained:  0.31 ± 0.02
    Improvement: +0.17
  DiffT:
    Scratch:     0.32 ± 0.06
    Pretrained:  0.53 ± 0.02
    Improvement: +0.21

10% of training data:
------------------------------------------------------------
  ATAT:
    Scratch:     0.43 ± 0.01
    Pretrained:  0.57 ± 0.01
    Improvement: +0.14
  DiffT:
    Scratch:     0.66 ± 0.01
    Pretrained:  0.74 ± 0.01
    Improvement: +0.08

25% of training data:
------------------------------------------------------------
  ATAT:
    Scratch:     0.53 ± 0.01
    Pretrained:  0.62 ± 0.01
    Improvement: +0.09
  DiffT:
    Scratch:     0.73 ± 0.00
    Pretrained:  0.78 ± 0.01
    Improvement: +0.05

50% of training data:
------------------------------------------------------------
  ATAT:
    Scratch:     0.62 ± 0.01
    Pretrained:  0.66 ± 0.01
    Impr

## Model Comparison: ATAT vs DiffT by Data Percentage

Compare ATAT (baseline) vs DiffT for scratch and pretrained settings.

In [25]:
# Compare ATAT vs DiffT for each training type at each percentage
if not df_all.empty:
    print("\nATAT vs DiffT Comparison by Data Percentage:")
    print("="*80)
    
    for pct in sorted(grouped['percentage'].unique()):
        pct_label = f"{int(pct*100)}%" if pct < 1.0 else "100%"
        print(f"\n{pct_label} of training data:")
        print("-" * 60)
        
        for train_type in ['scratch', 'pretrained', 'linear']:
            atat_data = grouped[
                (grouped['model'] == 'ATAT') & 
                (grouped['type'] == train_type) & 
                (grouped['percentage'] == pct)
            ]
            
            difft_data = grouped[
                (grouped['model'] == 'DiffT') & 
                (grouped['type'] == train_type) & 
                (grouped['percentage'] == pct)
            ]
            
            if not atat_data.empty and not difft_data.empty:
                atat_mean = atat_data.iloc[0]['mean']
                atat_std = atat_data.iloc[0]['std']
                difft_mean = difft_data.iloc[0]['mean']
                difft_std = difft_data.iloc[0]['std']
                diff = difft_mean - atat_mean
                
                print(f"  {train_type.capitalize()}:")
                print(f"    ATAT (baseline): {atat_mean:.2f} ± {atat_std:.2f}")
                print(f"    DiffT:           {difft_mean:.2f} ± {difft_std:.2f}")
                print(f"    Difference:      {diff:+.2f}")
            elif not atat_data.empty:
                print(f"  {train_type.capitalize()}: Only ATAT data available")
            elif not difft_data.empty:
                print(f"  {train_type.capitalize()}: Only DiffT data available")


ATAT vs DiffT Comparison by Data Percentage:

1% of training data:
------------------------------------------------------------
  Scratch:
    ATAT (baseline): 0.15 ± 0.05
    DiffT:           0.32 ± 0.06
    Difference:      +0.17
  Pretrained:
    ATAT (baseline): 0.31 ± 0.02
    DiffT:           0.53 ± 0.02
    Difference:      +0.21

10% of training data:
------------------------------------------------------------
  Scratch:
    ATAT (baseline): 0.43 ± 0.01
    DiffT:           0.66 ± 0.01
    Difference:      +0.23
  Pretrained:
    ATAT (baseline): 0.57 ± 0.01
    DiffT:           0.74 ± 0.01
    Difference:      +0.17

25% of training data:
------------------------------------------------------------
  Scratch:
    ATAT (baseline): 0.53 ± 0.01
    DiffT:           0.73 ± 0.00
    Difference:      +0.20
  Pretrained:
    ATAT (baseline): 0.62 ± 0.01
    DiffT:           0.78 ± 0.01
    Difference:      +0.16

50% of training data:
-----------------------------------------------

## Statistical Analysis: Permutation Tests

Perform permutation tests to compare models and training methods.

In [26]:
# Permutation tests: ATAT vs DiffT and Scratch vs Pretrained (paired)
try:
    from mlxtend.evaluate import permutation_test
except Exception:
    print("mlxtend not installed. Please install with: pip install mlxtend")
    permutation_test = None

if not df_all.empty and permutation_test is not None:
    perm_results = []

    def run_paired_test(a, b, desc, func='x_mean > y_mean'):
        try:
            return permutation_test(
                a,
                b,
                func=func,
                method='approximate',
                num_rounds=10000,
                seed=42,
            )
        except Exception as e:
            print(f"  {desc}: Failed - {e}")
            return None

    for pct in sorted(df_all['percentage'].unique()):
        pct_label = f"{int(pct*100)}%" if pct < 1.0 else "100%"
        print(f"\nPermutation tests for {pct_label} data:")
        print("="*60)

        # Test 1: ATAT vs DiffT for each training type (direction: DiffT > ATAT)
        for train_type in ['scratch', 'pretrained', 'linear']:
            atat_runs = df_all[
                (df_all['model'] == 'ATAT') &
                (df_all['type'] == train_type) &
                (df_all['percentage'] == pct)
            ]['test_f1'].values

            difft_runs = df_all[
                (df_all['model'] == 'DiffT') &
                (df_all['type'] == train_type) &
                (df_all['percentage'] == pct)
            ]['test_f1'].values

            pval = run_paired_test(atat_runs, difft_runs, f"ATAT vs DiffT ({train_type})", func='x_mean < y_mean')
            if pval is not None:
                perm_results.append({
                    'percentage': pct,
                    'comparison': f'ATAT vs DiffT ({train_type})',
                    'p_value': pval,
                    'atat_mean': atat_runs.mean(),
                    'difft_mean': difft_runs.mean()
                })
                print(f"  ATAT vs DiffT ({train_type}): p={pval:.4f}")

        # Test 2: Scratch vs Pretrained for each model (direction: Pretrained > Scratch)
        for model in ['ATAT', 'DiffT']:
            scratch_runs = df_all[
                (df_all['model'] == model) &
                (df_all['type'] == 'scratch') &
                (df_all['percentage'] == pct)
            ]['test_f1'].values

            pretrained_runs = df_all[
                (df_all['model'] == model) &
                (df_all['type'] == 'pretrained') &
                (df_all['percentage'] == pct)
            ]['test_f1'].values

            pval = run_paired_test(scratch_runs, pretrained_runs, f"{model} Scratch vs Pretrained", func='x_mean < y_mean')
            if pval is not None:
                perm_results.append({
                    'percentage': pct,
                    'comparison': f'{model} Scratch vs Pretrained',
                    'p_value': pval,
                    'scratch_mean': scratch_runs.mean(),
                    'pretrained_mean': pretrained_runs.mean()
                })
                print(f"  {model} Scratch vs Pretrained: p={pval:.4f}")

    perm_df = pd.DataFrame(perm_results)
    perm_df.to_csv("paper_notebooks/csv/model_comparison_permutation_tests.csv", index=False)
    print(f"\n\nSaved permutation test results to paper_notebooks/csv/model_comparison_permutation_tests.csv")
    display(perm_df)
elif permutation_test is None:
    print("Permutation test unavailable; install mlxtend and re-run.")
    perm_df = pd.DataFrame()
else:
    print("No data for permutation tests.")
    perm_df = pd.DataFrame()


Permutation tests for 1% data:
  ATAT vs DiffT (scratch): p=0.0053
  ATAT vs DiffT (pretrained): p=0.0001
  ATAT vs DiffT (linear): p=0.0001
  ATAT Scratch vs Pretrained: p=0.0001
  DiffT Scratch vs Pretrained: p=0.0001

Permutation tests for 10% data:
  ATAT vs DiffT (scratch): p=0.0053
  ATAT vs DiffT (pretrained): p=0.0001
  ATAT vs DiffT (linear): p=0.0001
  ATAT Scratch vs Pretrained: p=0.0001
  DiffT Scratch vs Pretrained: p=0.0001

Permutation tests for 25% data:
  ATAT vs DiffT (scratch): p=0.0053
  ATAT vs DiffT (pretrained): p=0.0001
  ATAT vs DiffT (linear): p=0.0001
  ATAT Scratch vs Pretrained: p=0.0001
  DiffT Scratch vs Pretrained: p=0.0001

Permutation tests for 50% data:
  ATAT vs DiffT (scratch): p=0.0053
  ATAT vs DiffT (pretrained): p=0.0001
  ATAT vs DiffT (linear): p=0.0001
  ATAT Scratch vs Pretrained: p=0.0001
  DiffT Scratch vs Pretrained: p=0.0001

Permutation tests for 100% data:
  ATAT vs DiffT (scratch): p=0.0053
  ATAT vs DiffT (pretrained): p=0.0001
  AT

,percentage,comparison,p_value,atat_mean,difft_mean,scratch_mean,pretrained_mean
0,0.01,ATAT vs DiffT (scratch),0.005299,0.146507,0.320049,NaN,NaN
1,0.01,ATAT vs DiffT (pretrained),0.000100,0.311564,0.526236,NaN,NaN
2,0.01,ATAT vs DiffT (linear),0.000100,NaN,NaN,NaN,NaN
3,0.01,ATAT Scratch vs Pretrained,0.000100,NaN,NaN,0.146507,0.311564
4,0.01,DiffT Scratch vs Pretrained,0.000100,NaN,NaN,0.320049,0.526236
5,0.10,ATAT vs DiffT (scratch),0.005299,0.432796,0.659324,NaN,NaN
6,0.10,ATAT vs DiffT (pretrained),0.000100,0.568001,0.738540,NaN,NaN
7,0.10,ATAT vs DiffT (linear),0.000100,NaN,NaN,NaN,NaN
8,0.10,ATAT Scratch vs Pretrained,0.000100,NaN,NaN,0.432796,0.568001
9,0.10,DiffT Scratch vs Pretrained,0.000100,NaN,NaN,0.659324,0.738540


## LaTeX Table with Statistical Significance

Generate publication-ready table with:
- Bold: Not significantly different from ATAT baseline (p > 0.05)
- † (dagger): Pretrained significantly better than scratch (p < 0.05)
- Linear probing only shown for 100% data

In [27]:
# Generate LaTeX table (original layout) with significance; linear shown as its own percentage row
if not df_all.empty:
    percentages = sorted(grouped[grouped['type'] != 'linear']['percentage'].unique())
    linear_rows = sorted(grouped[grouped['type'] == 'linear']['percentage'].unique())

    def diff_sig(pct, train_type):
        """Check if DiffT is significantly different from ATAT for given type"""
        if 'perm_df' in globals() and not perm_df.empty:
            res = perm_df[
                (perm_df['percentage'] == pct) &
                (perm_df['comparison'] == f'ATAT vs DiffT ({train_type})')
            ]
            if not res.empty:
                return res.iloc[0]['p_value'] < 0.05
        return False
    
    def pretrain_better_than_scratch(model, pct):
        """Check if pretrained is significantly better than scratch for given model"""
        if 'perm_df' in globals() and not perm_df.empty:
            res = perm_df[
                (perm_df['percentage'] == pct) &
                (perm_df['comparison'] == f'{model} Scratch vs Pretrained')
            ]
            if not res.empty:
                return res.iloc[0]['p_value'] < 0.05
        return False
    
    latex_lines = []
    latex_lines.append(r"\begin{table}[h]")
    latex_lines.append(r"\centering")
    latex_lines.append(r"\caption{Test F1 Performance Comparison (mean $\pm$ std). Bold: pretrained significantly better than scratch. $^\dagger$: DiffT scratch significantly different from ATAT scratch (p $<$ 0.05).}")
    latex_lines.append(r"\label{tab:model_comparison}")
    latex_lines.append(r"\begin{tabular}{|c|cc|cc|}")
    latex_lines.append(r"\hline")
    latex_lines.append(r"\multirow{2}{*}{\% Data} & \multicolumn{2}{c|}{ATAT (baseline)} & \multicolumn{2}{c|}{DiffT} \\")
    latex_lines.append(r"\cline{2-5}")
    latex_lines.append(r" & Scratch & Pretrained & Scratch & Pretrained \\")
    latex_lines.append(r"\hline")

    def build_cell(model, train_type, pct):
        subset = grouped[
            (grouped['model'] == model) &
            (grouped['type'] == train_type) &
            (grouped['percentage'] == pct)
        ]
        if subset.empty:
            return "-"
        mean_val = subset.iloc[0]['mean']
        std_val = subset.iloc[0]['std']
        cell = f"{mean_val:.3f} $\\pm$ {std_val:.3f}"
        
        # Add dagger if DiffT scratch is significantly different from ATAT scratch
        if model == 'DiffT' and train_type == 'scratch' and diff_sig(pct, 'scratch'):
            cell = f"{cell}$^\\dagger$"
        
        # Bold if pretrained is significantly better than scratch for this model
        if train_type == 'pretrained' and pretrain_better_than_scratch(model, pct):
            cell = f"\\textbf{{{cell}}}"
        return cell

    # Standard percentages
    for pct in percentages:
        pct_label = f"{int(pct*100)}" if pct < 1.0 else "100"
        row_data = [pct_label]
        for model in ['ATAT', 'DiffT']:
            row_data.append(build_cell(model, 'scratch', pct))
            row_data.append(build_cell(model, 'pretrained', pct))
        latex_lines.append(" & ".join(row_data) + r" \\")

    # Linear rows as percentage rows (value in Pretrained column)
    for pct in linear_rows:
        pct_label = f"{int(pct*100)} (Linear)" if pct < 1.0 else "100 (Linear)"
        row_data = [pct_label]
        for model in ['ATAT', 'DiffT']:
            row_data.append("-")
            row_data.append(build_cell(model, 'linear', pct))
        latex_lines.append(" & ".join(row_data) + r" \\")

    latex_lines.append(r"\hline")
    latex_lines.append(r"\end{tabular}")
    latex_lines.append(r"\end{table}")
    
    latex_table = "\n".join(latex_lines)
    print("\nLaTeX Table (original layout) with significance:")
    print("="*80)
    print(latex_table)
    
    latex_output = "paper_notebooks/model_comparison_table.tex"
    with open(latex_output, 'w') as f:
        f.write(latex_table)
    print(f"\nSaved LaTeX table to: {latex_output}")
else:
    print("Cannot generate table: missing data.")


LaTeX Table (original layout) with significance:
\begin{table}[h]
\centering
\caption{Test F1 Performance Comparison (mean $\pm$ std). Bold: pretrained significantly better than scratch. $^\dagger$: DiffT scratch significantly different from ATAT scratch (p $<$ 0.05).}
\label{tab:model_comparison}
\begin{tabular}{|c|cc|cc|}
\hline
\multirow{2}{*}{\% Data} & \multicolumn{2}{c|}{ATAT (baseline)} & \multicolumn{2}{c|}{DiffT} \\
\cline{2-5}
 & Scratch & Pretrained & Scratch & Pretrained \\
\hline
1 & 0.147 $\pm$ 0.048 & \textbf{0.312 $\pm$ 0.020} & 0.320 $\pm$ 0.056$^\dagger$ & \textbf{0.526 $\pm$ 0.019} \\
10 & 0.433 $\pm$ 0.011 & \textbf{0.568 $\pm$ 0.009} & 0.659 $\pm$ 0.011$^\dagger$ & \textbf{0.739 $\pm$ 0.006} \\
25 & 0.531 $\pm$ 0.010 & \textbf{0.620 $\pm$ 0.008} & 0.731 $\pm$ 0.005$^\dagger$ & \textbf{0.782 $\pm$ 0.005} \\
50 & 0.620 $\pm$ 0.009 & \textbf{0.659 $\pm$ 0.006} & 0.776 $\pm$ 0.005$^\dagger$ & \textbf{0.811 $\pm$ 0.004} \\
100 & 0.684 $\pm$ 0.006 & \textbf{0.691 $\pm$ 

# Multimodal Model Comparison

Analyze multimodal experiments that include tabular features alongside light curve data.

In [28]:
# Multimodal experiment groups
multimodal_groups = {
    'ATAT-Multimodal-Scratch': {'model': 'ATAT', 'type': 'scratch', 'group': 'ATAT-long-Percentages-Multimodal'},
    'ATAT-Multimodal-PT': {'model': 'ATAT', 'type': 'pretrained', 'group': 'ATAT-long-Percentages-PT-Multimodal'},
    'DiffT-Multimodal-Scratch': {'model': 'DiffT', 'type': 'scratch', 'group': 'DiffT-Long-Percentages-Multimodal'},
    'DiffT-Multimodal-PT': {'model': 'DiffT', 'type': 'pretrained', 'group': 'DiffT-Long-Percentages-PT-Multimodal'},
}

print(f"Configured {len(multimodal_groups)} multimodal experiment groups")

Configured 4 multimodal experiment groups


In [29]:
# Fetch multimodal runs
all_multimodal_data = []

for exp_name, exp_config in multimodal_groups.items():
    group_name = exp_config['group']
    model_type = exp_config['model']
    train_type = exp_config['type']
    
    print(f"\nFetching: {exp_name} (Group: {group_name})...")
    
    try:
        runs = api.runs(project_path, filters={"group": group_name, "state": "finished"})
        print(f"  Found {len(runs)} finished runs")
        
        for run in runs:
            cfg = {k: v for k, v in run.config.items() if not str(k).startswith("_")}
            summary = run.summary._json_dict
            
            # Extract percentage and split
            percentage = get_cfg(cfg, "data.percentage", ["data", "percentage"], 1.0)
            split = get_cfg(cfg, "data.split", ["data", "split"], 0)
            
            # Extract test F1
            test_f1 = None
            for key in ["test/f1", "test/f1_macro", "best_f1"]:
                if key in summary and isinstance(summary[key], (int, float)):
                    test_f1 = float(summary[key])
                    break
            
            if test_f1 is not None:
                all_multimodal_data.append({
                    'experiment': exp_name,
                    'model': model_type,
                    'type': train_type,
                    'percentage': percentage,
                    'split': split,
                    'test_f1': test_f1,
                    'run_name': run.name
                })
    
    except Exception as e:
        print(f"  Error fetching {group_name}: {e}")

# Build multimodal DataFrame
df_multimodal = pd.DataFrame(all_multimodal_data)
raw_count_mm = len(df_multimodal)
df_multimodal['test_f1'] = pd.to_numeric(df_multimodal['test_f1'], errors='coerce')
df_multimodal = df_multimodal.dropna(subset=['test_f1'])
print(f"\nDropped {raw_count_mm - len(df_multimodal)} runs with missing/non-numeric test_f1.")

df_multimodal.to_csv("paper_notebooks/csv/model_comparison_multimodal_test_f1_raw.csv", index=False)
print(f"Saved raw multimodal run-level F1s to paper_notebooks/csv/model_comparison_multimodal_test_f1_raw.csv")

print(f"\n{'='*60}")
print(f"Total multimodal runs collected: {len(df_multimodal)}")
print(f"\nRuns per experiment:")
print(df_multimodal.groupby('experiment').size())


Fetching: ATAT-Multimodal-Scratch (Group: ATAT-long-Percentages-Multimodal)...


  Found 25 finished runs

Fetching: ATAT-Multimodal-PT (Group: ATAT-long-Percentages-PT-Multimodal)...
  Found 125 finished runs

Fetching: DiffT-Multimodal-Scratch (Group: DiffT-Long-Percentages-Multimodal)...
  Found 25 finished runs

Fetching: DiffT-Multimodal-PT (Group: DiffT-Long-Percentages-PT-Multimodal)...
  Found 125 finished runs

Dropped 0 runs with missing/non-numeric test_f1.
Saved raw multimodal run-level F1s to paper_notebooks/csv/model_comparison_multimodal_test_f1_raw.csv

Total multimodal runs collected: 300

Runs per experiment:
experiment
ATAT-Multimodal-PT          125
ATAT-Multimodal-Scratch      25
DiffT-Multimodal-PT         125
DiffT-Multimodal-Scratch     25
dtype: int64


In [30]:
# Aggregate multimodal results
if not df_multimodal.empty:
    grouped_mm = df_multimodal.groupby(['model', 'type', 'percentage'])['test_f1'].agg([
        ('mean', 'mean'),
        ('std', 'std'),
        ('count', 'count')
    ]).reset_index()
    
    # Sort for display
    grouped_mm = grouped_mm.sort_values(['percentage', 'model', 'type'])
    
    print("Aggregated Multimodal Results (Test F1):")
    print("="*80)
    display(grouped_mm)
    
    # Save to CSV
    output_path_mm = "paper_notebooks/csv/model_comparison_multimodal_test_f1.csv"
    grouped_mm.to_csv(output_path_mm, index=False)
    print(f"\nSaved to: {output_path_mm}")
else:
    print("No multimodal data to aggregate.")

Aggregated Multimodal Results (Test F1):


,model,type,percentage,mean,std,count
0,ATAT,pretrained,0.01,0.427602,0.023460,25
5,ATAT,scratch,0.01,0.403607,0.037142,5
10,DiffT,pretrained,0.01,0.561511,0.014402,25
15,DiffT,scratch,0.01,0.519110,0.031483,5
1,ATAT,pretrained,0.10,0.781563,0.004786,25
6,ATAT,scratch,0.10,0.758704,0.007628,5
11,DiffT,pretrained,0.10,0.804258,0.008861,25
16,DiffT,scratch,0.10,0.801127,0.003419,5
2,ATAT,pretrained,0.25,0.827396,0.003787,25
7,ATAT,scratch,0.25,0.812715,0.003386,5



Saved to: paper_notebooks/csv/model_comparison_multimodal_test_f1.csv


In [31]:
# Create multimodal pivot table
if not df_multimodal.empty:
    grouped_mm['f1_formatted'] = grouped_mm.apply(
        lambda row: f"{row['mean']:.3f} ± {row['std']:.3f}", axis=1
    )
    
    pivot_mm = grouped_mm.pivot_table(
        index='percentage',
        columns=['model', 'type'],
        values='f1_formatted',
        aggfunc='first'
    )
    
    # Relabel index for display
    pivot_mm.index = [f"{int(p*100)}" if p < 1.0 else "100" for p in pivot_mm.index]
    
    print("\nMultimodal Pivot Table: Test F1 (mean ± std) by Percentage and Model")
    print("="*80)
    display(pivot_mm)
    
    pivot_output_mm = "paper_notebooks/csv/model_comparison_multimodal_pivot.csv"
    pivot_mm.to_csv(pivot_output_mm)
    print(f"\nSaved multimodal pivot table to: {pivot_output_mm}")


Multimodal Pivot Table: Test F1 (mean ± std) by Percentage and Model


model           ATAT                         DiffT               
type      pretrained        scratch     pretrained        scratch
1      0.428 ± 0.023  0.404 ± 0.037  0.562 ± 0.014  0.519 ± 0.031
10     0.782 ± 0.005  0.759 ± 0.008  0.804 ± 0.009  0.801 ± 0.003
25     0.827 ± 0.004  0.813 ± 0.003  0.846 ± 0.005  0.834 ± 0.003
50     0.850 ± 0.004  0.840 ± 0.003  0.863 ± 0.004  0.857 ± 0.004
100    0.869 ± 0.003  0.866 ± 0.003  0.884 ± 0.003  0.880 ± 0.002


Saved multimodal pivot table to: paper_notebooks/csv/model_comparison_multimodal_pivot.csv


In [32]:
# Generate multimodal LaTeX table
if not df_multimodal.empty:
    percentages_mm = sorted(grouped_mm['percentage'].unique())
    
    latex_lines_mm = []
    latex_lines_mm.append(r"\begin{table}[h]")
    latex_lines_mm.append(r"\centering")
    latex_lines_mm.append(r"\caption{Multimodal Test F1 Performance Comparison (mean $\pm$ std). ATAT is the baseline model.}")
    latex_lines_mm.append(r"\label{tab:multimodal_comparison}")
    latex_lines_mm.append(r"\begin{tabular}{|c|cc|cc|}")
    latex_lines_mm.append(r"\hline")
    latex_lines_mm.append(r"\multirow{2}{*}{\% Data} & \multicolumn{2}{c|}{ATAT (baseline)} & \multicolumn{2}{c|}{DiffT} \\")
    latex_lines_mm.append(r"\cline{2-5}")
    latex_lines_mm.append(r" & Scratch & Pretrained & Scratch & Pretrained \\")
    latex_lines_mm.append(r"\hline")
    
    def get_cell_mm(model, train_type, pct):
        subset = grouped_mm[
            (grouped_mm['model'] == model) &
            (grouped_mm['type'] == train_type) &
            (grouped_mm['percentage'] == pct)
        ]
        if subset.empty:
            return "-"
        mean_val = subset.iloc[0]['mean']
        std_val = subset.iloc[0]['std']
        return f"{mean_val:.3f} $\\pm$ {std_val:.3f}"
    
    for pct in percentages_mm:
        pct_label = f"{int(pct*100)}" if pct < 1.0 else "100"
        row_data = [pct_label]
        for model in ['ATAT', 'DiffT']:
            row_data.append(get_cell_mm(model, 'scratch', pct))
            row_data.append(get_cell_mm(model, 'pretrained', pct))
        latex_lines_mm.append(" & ".join(row_data) + r" \\")
    
    latex_lines_mm.append(r"\hline")
    latex_lines_mm.append(r"\end{tabular}")
    latex_lines_mm.append(r"\end{table}")
    
    latex_table_mm = "\n".join(latex_lines_mm)
    print("\nMultimodal LaTeX Table:")
    print("="*80)
    print(latex_table_mm)
    
    latex_output_mm = "paper_notebooks/model_comparison_multimodal_table.tex"
    with open(latex_output_mm, 'w') as f:
        f.write(latex_table_mm)
    print(f"\nSaved multimodal LaTeX table to: {latex_output_mm}")


Multimodal LaTeX Table:
\begin{table}[h]
\centering
\caption{Multimodal Test F1 Performance Comparison (mean $\pm$ std). ATAT is the baseline model.}
\label{tab:multimodal_comparison}
\begin{tabular}{|c|cc|cc|}
\hline
\multirow{2}{*}{\% Data} & \multicolumn{2}{c|}{ATAT (baseline)} & \multicolumn{2}{c|}{DiffT} \\
\cline{2-5}
 & Scratch & Pretrained & Scratch & Pretrained \\
\hline
1 & 0.404 $\pm$ 0.037 & 0.428 $\pm$ 0.023 & 0.519 $\pm$ 0.031 & 0.562 $\pm$ 0.014 \\
10 & 0.759 $\pm$ 0.008 & 0.782 $\pm$ 0.005 & 0.801 $\pm$ 0.003 & 0.804 $\pm$ 0.009 \\
25 & 0.813 $\pm$ 0.003 & 0.827 $\pm$ 0.004 & 0.834 $\pm$ 0.003 & 0.846 $\pm$ 0.005 \\
50 & 0.840 $\pm$ 0.003 & 0.850 $\pm$ 0.004 & 0.857 $\pm$ 0.004 & 0.863 $\pm$ 0.004 \\
100 & 0.866 $\pm$ 0.003 & 0.869 $\pm$ 0.003 & 0.880 $\pm$ 0.002 & 0.884 $\pm$ 0.003 \\
\hline
\end{tabular}
\end{table}

Saved multimodal LaTeX table to: paper_notebooks/model_comparison_multimodal_table.tex


In [33]:
# Multimodal permutation tests
if not df_multimodal.empty and permutation_test is not None:
    perm_results_mm = []
    
    for pct in sorted(df_multimodal['percentage'].unique()):
        pct_label = f"{int(pct*100)}%" if pct < 1.0 else "100%"
        print(f"\nMultimodal permutation tests for {pct_label} data:")
        print("="*60)
        
        # Test 1: ATAT vs DiffT for each training type
        for train_type in ['scratch', 'pretrained']:
            atat_runs = df_multimodal[
                (df_multimodal['model'] == 'ATAT') &
                (df_multimodal['type'] == train_type) &
                (df_multimodal['percentage'] == pct)
            ]['test_f1'].values
            
            difft_runs = df_multimodal[
                (df_multimodal['model'] == 'DiffT') &
                (df_multimodal['type'] == train_type) &
                (df_multimodal['percentage'] == pct)
            ]['test_f1'].values
            
            pval = run_paired_test(atat_runs, difft_runs, f"ATAT vs DiffT ({train_type})", func='x_mean < y_mean')
            if pval is not None:
                perm_results_mm.append({
                    'percentage': pct,
                    'comparison': f'ATAT vs DiffT ({train_type})',
                    'p_value': pval,
                    'atat_mean': atat_runs.mean(),
                    'difft_mean': difft_runs.mean()
                })
                print(f"  ATAT vs DiffT ({train_type}): p={pval:.4f}")
        
        # Test 2: Scratch vs Pretrained for each model
        for model in ['ATAT', 'DiffT']:
            scratch_runs = df_multimodal[
                (df_multimodal['model'] == model) &
                (df_multimodal['type'] == 'scratch') &
                (df_multimodal['percentage'] == pct)
            ]['test_f1'].values
            
            pretrained_runs = df_multimodal[
                (df_multimodal['model'] == model) &
                (df_multimodal['type'] == 'pretrained') &
                (df_multimodal['percentage'] == pct)
            ]['test_f1'].values
            
            pval = run_paired_test(scratch_runs, pretrained_runs, f"{model} Scratch vs Pretrained", func='x_mean < y_mean')
            if pval is not None:
                perm_results_mm.append({
                    'percentage': pct,
                    'comparison': f'{model} Scratch vs Pretrained',
                    'p_value': pval,
                    'scratch_mean': scratch_runs.mean(),
                    'pretrained_mean': pretrained_runs.mean()
                })
                print(f"  {model} Scratch vs Pretrained: p={pval:.4f}")
    
    perm_df_mm = pd.DataFrame(perm_results_mm)
    perm_df_mm.to_csv("paper_notebooks/csv/model_comparison_multimodal_permutation_tests.csv", index=False)
    print(f"\n\nSaved multimodal permutation test results to paper_notebooks/csv/model_comparison_multimodal_permutation_tests.csv")
    display(perm_df_mm)
elif permutation_test is None:
    print("Permutation test unavailable; install mlxtend and re-run.")
    perm_df_mm = pd.DataFrame()
else:
    print("No multimodal data for permutation tests.")
    perm_df_mm = pd.DataFrame()


Multimodal permutation tests for 1% data:


  ATAT vs DiffT (scratch): p=0.0053
  ATAT vs DiffT (pretrained): p=0.0001
  ATAT Scratch vs Pretrained: p=0.0379
  DiffT Scratch vs Pretrained: p=0.0002

Multimodal permutation tests for 10% data:
  ATAT vs DiffT (scratch): p=0.0053
  ATAT vs DiffT (pretrained): p=0.0001
  ATAT Scratch vs Pretrained: p=0.0001
  DiffT Scratch vs Pretrained: p=0.2279

Multimodal permutation tests for 25% data:
  ATAT vs DiffT (scratch): p=0.0053
  ATAT vs DiffT (pretrained): p=0.0001
  ATAT Scratch vs Pretrained: p=0.0001
  DiffT Scratch vs Pretrained: p=0.0001

Multimodal permutation tests for 50% data:
  ATAT vs DiffT (scratch): p=0.0053
  ATAT vs DiffT (pretrained): p=0.0001
  ATAT Scratch vs Pretrained: p=0.0001
  DiffT Scratch vs Pretrained: p=0.0049

Multimodal permutation tests for 100% data:
  ATAT vs DiffT (scratch): p=0.0053
  ATAT vs DiffT (pretrained): p=0.0001
  ATAT Scratch vs Pretrained: p=0.0364
  DiffT Scratch vs Pretrained: p=0.0004


Saved multimodal permutation test results to paper_

,percentage,comparison,p_value,atat_mean,difft_mean,scratch_mean,pretrained_mean
0,0.01,ATAT vs DiffT (scratch),0.005299,0.403607,0.519110,NaN,NaN
1,0.01,ATAT vs DiffT (pretrained),0.000100,0.427602,0.561511,NaN,NaN
2,0.01,ATAT Scratch vs Pretrained,0.037896,NaN,NaN,0.403607,0.427602
3,0.01,DiffT Scratch vs Pretrained,0.000200,NaN,NaN,0.519110,0.561511
4,0.10,ATAT vs DiffT (scratch),0.005299,0.758704,0.801127,NaN,NaN
5,0.10,ATAT vs DiffT (pretrained),0.000100,0.781563,0.804258,NaN,NaN
6,0.10,ATAT Scratch vs Pretrained,0.000100,NaN,NaN,0.758704,0.781563
7,0.10,DiffT Scratch vs Pretrained,0.227877,NaN,NaN,0.801127,0.804258
8,0.25,ATAT vs DiffT (scratch),0.005299,0.812715,0.833733,NaN,NaN
9,0.25,ATAT vs DiffT (pretrained),0.000100,0.827396,0.845928,NaN,NaN


In [34]:
# Generate multimodal LaTeX table with significance
if not df_multimodal.empty:
    percentages_mm = sorted(grouped_mm['percentage'].unique())
    
    def diff_sig_mm(pct, train_type):
        """Check if DiffT is significantly different from ATAT for given type"""
        if 'perm_df_mm' in globals() and not perm_df_mm.empty:
            res = perm_df_mm[
                (perm_df_mm['percentage'] == pct) &
                (perm_df_mm['comparison'] == f'ATAT vs DiffT ({train_type})')
            ]
            if not res.empty:
                return res.iloc[0]['p_value'] < 0.05
        return False
    
    def pretrain_better_than_scratch_mm(model, pct):
        """Check if pretrained is significantly better than scratch for given model"""
        if 'perm_df_mm' in globals() and not perm_df_mm.empty:
            res = perm_df_mm[
                (perm_df_mm['percentage'] == pct) &
                (perm_df_mm['comparison'] == f'{model} Scratch vs Pretrained')
            ]
            if not res.empty:
                return res.iloc[0]['p_value'] < 0.05
        return False
    
    latex_lines_mm_sig = []
    latex_lines_mm_sig.append(r"\begin{table}[h]")
    latex_lines_mm_sig.append(r"\centering")
    latex_lines_mm_sig.append(r"\caption{Multimodal Test F1 Performance Comparison (mean $\pm$ std). Bold: pretrained significantly better than scratch. $^\dagger$: DiffT scratch significantly different from ATAT scratch (p $<$ 0.05).}")
    latex_lines_mm_sig.append(r"\label{tab:multimodal_comparison_sig}")
    latex_lines_mm_sig.append(r"\begin{tabular}{|c|cc|cc|}")
    latex_lines_mm_sig.append(r"\hline")
    latex_lines_mm_sig.append(r"\multirow{2}{*}{\% Data} & \multicolumn{2}{c|}{ATAT (baseline)} & \multicolumn{2}{c|}{DiffT} \\")
    latex_lines_mm_sig.append(r"\cline{2-5}")
    latex_lines_mm_sig.append(r" & Scratch & Pretrained & Scratch & Pretrained \\")
    latex_lines_mm_sig.append(r"\hline")
    
    def build_cell_mm(model, train_type, pct):
        subset = grouped_mm[
            (grouped_mm['model'] == model) &
            (grouped_mm['type'] == train_type) &
            (grouped_mm['percentage'] == pct)
        ]
        if subset.empty:
            return "-"
        mean_val = subset.iloc[0]['mean']
        std_val = subset.iloc[0]['std']
        cell = f"{mean_val:.3f} $\\pm$ {std_val:.3f}"
        
        # Add dagger if DiffT scratch is significantly different from ATAT scratch
        if model == 'DiffT' and train_type == 'scratch' and diff_sig_mm(pct, 'scratch'):
            cell = f"{cell}$^\\dagger$"
        
        # Bold if pretrained is significantly better than scratch for this model
        if train_type == 'pretrained' and pretrain_better_than_scratch_mm(model, pct):
            cell = f"\\textbf{{{cell}}}"
            
        return cell
    
    for pct in percentages_mm:
        pct_label = f"{int(pct*100)}" if pct < 1.0 else "100"
        row_data = [pct_label]
        for model in ['ATAT', 'DiffT']:
            row_data.append(build_cell_mm(model, 'scratch', pct))
            row_data.append(build_cell_mm(model, 'pretrained', pct))
        latex_lines_mm_sig.append(" & ".join(row_data) + r" \\")
    
    latex_lines_mm_sig.append(r"\hline")
    latex_lines_mm_sig.append(r"\end{tabular}")
    latex_lines_mm_sig.append(r"\end{table}")
    
    latex_table_mm_sig = "\n".join(latex_lines_mm_sig)
    print("\nMultimodal LaTeX Table with Significance:")
    print("="*80)
    print(latex_table_mm_sig)
    
    latex_output_mm_sig = "paper_notebooks/model_comparison_multimodal_table_sig.tex"
    with open(latex_output_mm_sig, 'w') as f:
        f.write(latex_table_mm_sig)
    print(f"\nSaved multimodal LaTeX table with significance to: {latex_output_mm_sig}")
else:
    print("Cannot generate multimodal table: missing data.")


Multimodal LaTeX Table with Significance:
\begin{table}[h]
\centering
\caption{Multimodal Test F1 Performance Comparison (mean $\pm$ std). Bold: pretrained significantly better than scratch. $^\dagger$: DiffT scratch significantly different from ATAT scratch (p $<$ 0.05).}
\label{tab:multimodal_comparison_sig}
\begin{tabular}{|c|cc|cc|}
\hline
\multirow{2}{*}{\% Data} & \multicolumn{2}{c|}{ATAT (baseline)} & \multicolumn{2}{c|}{DiffT} \\
\cline{2-5}
 & Scratch & Pretrained & Scratch & Pretrained \\
\hline
1 & 0.404 $\pm$ 0.037 & \textbf{0.428 $\pm$ 0.023} & 0.519 $\pm$ 0.031$^\dagger$ & \textbf{0.562 $\pm$ 0.014} \\
10 & 0.759 $\pm$ 0.008 & \textbf{0.782 $\pm$ 0.005} & 0.801 $\pm$ 0.003$^\dagger$ & 0.804 $\pm$ 0.009 \\
25 & 0.813 $\pm$ 0.003 & \textbf{0.827 $\pm$ 0.004} & 0.834 $\pm$ 0.003$^\dagger$ & \textbf{0.846 $\pm$ 0.005} \\
50 & 0.840 $\pm$ 0.003 & \textbf{0.850 $\pm$ 0.004} & 0.857 $\pm$ 0.004$^\dagger$ & \textbf{0.863 $\pm$ 0.004} \\
100 & 0.866 $\pm$ 0.003 & \textbf{0.869 $\